# KMeans / PCA Diagnose
Separates Diagnose-Notebook f?r KMeans und PCA.


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, silhouette_score

from helper import load_geocsv, smooth_clusters_by_block

DEFAULT_CLUSTER_COUNT = 5
KMEANS_CLUSTER_COL = "cluster_kmeans_diagnose"
KMEANS_BLOCK_CLUSTER_COL = "cluster_kmeans_block_diagnose"
KMEANS_BLOCK_CHANGED_COL = "cluster_kmeans_block_changed_diagnose"
CLUSTER_FEATURE_WEIGHTS = {
    "z_centrality": 0.50,
    "z_einzelhandel_distance": 0.075,
    "z_einzelhandel_near_500": 0.125,
    "z_haltestelle_distance": 0.04,
    "z_haltestellen_count_within_500m": 0.04,
    "z_haltestellen_linien_count": 0.08,
    "z_laerm_index_tag": 0.07,
    "z_rail_penalty": 0.07,
}
z_vars = list(CLUSTER_FEATURE_WEIGHTS)
weighted_vars = [f"weighted_{col}" for col in z_vars]

gdf = load_geocsv("out/wohnlagen_brb_2026.csv")
for source_col, weight in CLUSTER_FEATURE_WEIGHTS.items():
    weighted_col = f"weighted_{source_col}"
    if weighted_col not in gdf.columns:
        gdf[weighted_col] = np.sqrt(weight) * gdf[source_col]


In [ ]:
mask_complete = gdf[weighted_vars].notna().all(axis=1)
X = gdf.loc[mask_complete, weighted_vars].to_numpy()
model = KMeans(n_clusters=DEFAULT_CLUSTER_COUNT, random_state=42).fit(X)
gdf.loc[mask_complete, KMEANS_CLUSTER_COL] = model.labels_
gdf[KMEANS_CLUSTER_COL] = gdf[KMEANS_CLUSTER_COL].astype("Int64")
gdf, kmeans_block_majority = smooth_clusters_by_block(
    gdf,
    source_cluster_col=KMEANS_CLUSTER_COL,
    target_cluster_col=KMEANS_BLOCK_CLUSTER_COL,
    changed_flag_col=KMEANS_BLOCK_CHANGED_COL,
)

silhouette_per_cluster = (
    pd.DataFrame({
        "Cluster": model.labels_,
        "silhouette_score": silhouette_samples(X, model.labels_),
    })
    .groupby("Cluster", as_index=False)
    .agg(anzahl=("silhouette_score", "size"), silhouette_score=("silhouette_score", "mean"))
)
display(silhouette_per_cluster.round(3))
display(kmeans_block_majority.head())
print("Silhouette gesamt:", round(silhouette_score(X, model.labels_), 3))


In [ ]:
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X)
print("Explained variance ratio:", np.round(pca.explained_variance_ratio_, 3))

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], c=model.labels_, s=8, cmap="tab10", alpha=0.7)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title(f"PCA-Projektion der KMeans-Cluster (k={DEFAULT_CLUSTER_COUNT})")
fig.colorbar(scatter, ax=ax, label="Cluster")
plt.tight_layout()
plt.show()
